# pythscribe — `@wasm` client-side in a Gradio app (no round-trip, ~10 ms)

The durable place `@wasm` wins is the **browser**, where CPython (and Numba/NumPy) don't run. This
notebook runs the *same* Python kernels **in the browser tab**, client-side, using **Gradio's built-in
`js=` event hook** — no custom component, no `WasmFunction`, no server round-trip. Measured latency
(bottom cell): **client-side `js=` ~10 ms** vs server ~60 ms vs the old `WasmFunction` path ~290 ms
(and ~0.1 ms in a plain static page with no Gradio at all).

How, with zero new components: `demo.load(None, None, None, js="async()=>{ const m = await
import('<bundle_url>'); window.__k = v => m.fn(...); }")` imports the compiled bundle into the page,
then `slider.change(None, [slider], [out], js="(v)=>window.__k(v)")` runs **client-side** — the input
never reaches the server, so it's also genuinely private (the PII tab's text stays in the tab).

Run in the **PythScribe (Gradio)** kernel (`pyths-gradio`).

## Kernels — four scalar use cases (browser) + two image filters (server)

In [1]:
from __future__ import annotations
import json, random, time
from pathlib import Path
import numpy as np
from PIL import Image
from pythscribe import wasm, binding_of

@wasm
def count_above(xs: list[float], thr: float) -> float:
    n = 0.0
    for x in xs:
        if x > thr:
            n = n + 1.0
    return n

@wasm
def luhn_ok(digits: list[int]) -> float:
    total = 0
    parity = len(digits) % 2
    for i in range(len(digits)):
        d = digits[i]
        if i % 2 == parity:
            d = d * 2
            if d > 9:
                d = d - 9
        total = total + d
    if total % 10 == 0:
        return 0.0
    return 1.0

@wasm
def pii_scan(text: list[int], min_run: int) -> float:
    runs = 0.0
    cur = 0
    for ch in text:
        if ch >= 48 and ch <= 57:
            cur = cur + 1
            if cur == min_run:
                runs = runs + 1.0
        else:
            cur = 0
    return runs

@wasm
def monthly_payment(principal: float, annual_rate_pct: float, months: int) -> float:
    r = annual_rate_pct / 1200.0
    if r == 0.0:
        return principal / months
    factor = 1.0
    base = 1.0 + r
    i = 0
    while i < months:
        factor = factor * base
        i = i + 1
    return principal * r * factor / (factor - 1.0)

@wasm
def threshold_lum(img: Array[uint8, 2], out: Array[uint8, 2], h: int, w: int, thr: int) -> int:
    for y in range(h):
        for x in range(w):
            c = x * 3
            s = img[y][c] + img[y][c + 1] + img[y][c + 2]
            v = 0
            if s > thr:
                v = 255
            out[y][c] = v; out[y][c + 1] = v; out[y][c + 2] = v
    return h * w

@wasm
def sobel(img: Array[uint8, 2], out: Array[uint8, 2], h: int, w: int) -> int:
    for y in range(1, h - 1):
        for x in range(1, w - 1):
            c = x * 3
            gx = img[y-1][c+3] + 2*img[y][c+3] + img[y+1][c+3] - img[y-1][c-3] - 2*img[y][c-3] - img[y+1][c-3]
            gy = img[y+1][c-3] + 2*img[y+1][c] + img[y+1][c+3] - img[y-1][c-3] - 2*img[y-1][c] - img[y-1][c+3]
            m = gx
            if m < 0:
                m = -m
            nn = gy
            if nn < 0:
                nn = -nn
            e = m + nn
            if e > 255:
                e = 255
            out[y][c] = e; out[y][c + 1] = e; out[y][c + 2] = e
    return (h - 2) * (w - 2)

def threshold_py(img, out, h, w, thr):
    for y in range(h):
        for x in range(w):
            c = x*3; s = int(img[y][c])+int(img[y][c+1])+int(img[y][c+2])
            v = 255 if s > thr else 0
            out[y][c]=v; out[y][c+1]=v; out[y][c+2]=v
    return h*w
def sobel_py(img, out, h, w):
    for y in range(1, h-1):
        for x in range(1, w-1):
            c=x*3
            gx=int(img[y-1][c+3])+2*int(img[y][c+3])+int(img[y+1][c+3])-int(img[y-1][c-3])-2*int(img[y][c-3])-int(img[y+1][c-3])
            gy=int(img[y+1][c-3])+2*int(img[y+1][c])+int(img[y+1][c+3])-int(img[y-1][c-3])-2*int(img[y-1][c])-int(img[y-1][c+3])
            e=abs(gx)+abs(gy)
            if e>255: e=255
            out[y][c]=e; out[y][c+1]=e; out[y][c+2]=e
    return (h-2)*(w-2)
print('kernels defined')

kernels defined


## One Gradio app — four browser tabs (`js=` client-side) + a server image tab

Drag the sliders / edit the text: each result is computed **in your browser** (the timing shown is the
`@wasm` in-tab compute; there is no server round-trip). The **Card validator** and **PII scan** never
send their input to the server. The **Image filters** tab is server-side (~50×) — image outputs
client-side need the typed-array browser adapter (the remaining v0.2.6 item).

## Placement ceiling — a runnable contrast (Streamlit vs Gradio-server vs client-side `@wasm`)

A common question: *"Streamlit re-runs the whole script on the server on every interaction — does this
get around that?"* The **Placement ceiling** tab in the app below answers it by putting the two
placements of the *same compute* (a monthly-payment slider) side by side:

- **Streamlit:** each widget change re-runs the whole script **on the server** — a full round-trip **and**
  a script re-execution per interaction.
- **Gradio native callback (`fn=`):** the change **round-trips to the server** once per interaction (no
  full-script re-run, but still a server hop) — the *right-hand* slider in the tab.
- **Compiled `@wasm` via `js=`:** the *same* compute recompiles to run **in the browser tab**, so the
  slider → recomputed-result loop runs client-side with **no server call** — the *left-hand* slider.

This is the placement-ceiling table's **"slider → recomputed result: runs in the browser"** row, made
runnable: drag both sliders and compare. It's a fair contrast, not a dunk — the native server path is
perfectly fine for plenty of apps; the point is that with `@wasm` the client-side placement is
*available* when you want it (latency, offline, or keeping the input in the tab).

In [ ]:
import gradio as gr
from pythscribe.gradio import bundle_url

# compile each kernel so its browser bundle URL resolves, then embed a client-side loader
count_above([0.1, 0.9], 0.5); luhn_ok([4, 2, 4, 2]); pii_scan([52, 53], 9); monthly_payment(1000.0, 5.0, 12)
F_URL = bundle_url(count_above); L_URL = bundle_url(luhn_ok); P_URL = bundle_url(pii_scan); C_URL = bundle_url(monthly_payment)
_rng = random.Random(7); DATA = [_rng.random() for _ in range(2000)]

LOAD_JS = ("async () => {\n  const [fa, la, pa, ca] = await Promise.all([import(\"F_URL\"), import(\"L_URL\"), import(\"P_URL\"), import(\"C_URL\")]);\n  window.__D = DATA_JSON;\n  window.__filter = (v) => { const t = performance.now(); const n = fa.count_above(window.__D, parseFloat(v));\n    return n + \" of \" + window.__D.length + \" above \" + parseFloat(v).toFixed(2) + \"  (@wasm \" + (performance.now()-t).toFixed(2) + \" ms, in your browser, no server)\"; };\n  window.__luhn = (card) => { const d = (card||\"\").split(\"\").filter(c => c>=\"0\" && c<=\"9\").map(Number);\n    if (!d.length) return \"enter a card number\"; const r = la.luhn_ok(d);\n    return (r === 0 ? \"VALID card\" : \"INVALID (fails Luhn)\") + \"  (checked in your browser; the number never left the tab)\"; };\n  window.__pii = (text) => { const b = Array.from(new TextEncoder().encode(text||\"\")); const n = pa.pii_scan(b, 9);\n    return (n ? n + \" sensitive item(s) - redact before upload\" : \"no long digit runs found\") + \"  (scanned in your browser; the text was never sent to the server)\"; };\n  window.__calc = (p, r, m) => { const v = ca.monthly_payment(parseFloat(p), parseFloat(r), parseInt(m));\n    return \"$\" + v.toLocaleString(\"en-US\", {minimumFractionDigits: 2, maximumFractionDigits: 2}) + \" / month  (computed in your browser)\"; };\n  window.__pc = (p) => { const t = performance.now(); const v = ca.monthly_payment(parseFloat(p), 6.0, 360);\n    return \"$\" + v.toLocaleString(\"en-US\", {minimumFractionDigits: 2, maximumFractionDigits: 2}) + \" / month  (@wasm \" + (performance.now()-t).toFixed(2) + \" ms, 0 server calls, in your browser)\"; };\n  window.__loaded = true;\n}"
    .replace('F_URL', F_URL).replace('L_URL', L_URL).replace('P_URL', P_URL).replace('C_URL', C_URL)
    .replace('DATA_JSON', json.dumps(DATA)))
def CJS(name): return '(...a) => (window.' + name + ' ? window.' + name + '(...a) : "loading...")'

FILT = {'threshold': (threshold_lum, threshold_py), 'sobel': (sobel, sobel_py)}
RGB0 = np.asarray(Image.open('../gradio-image-preprocess/test_images/photo_640x480.jpg').convert('RGB'), dtype=np.uint8)
def img_run(image, kind, thr, path):
    if image is None: return gr.skip(), ''
    rgb = np.ascontiguousarray(image, dtype=np.uint8); h, w = rgb.shape[:2]
    im = rgb.reshape(h, w*3); out = np.zeros_like(im); kw, kp = FILT[kind]; fn = kw if path == '@wasm' else kp
    t0 = time.perf_counter()
    fn(im, out, h, w, int(thr)*3) if kind == 'threshold' else fn(im, out, h, w)
    return out.reshape(h, w, 3), f'{path}: {(time.perf_counter()-t0)*1e3:.1f} ms (server-side)'

# plain-Python twin of the monthly_payment kernel, for the native server-callback slider (the same
# arithmetic the @wasm kernel does, but run on the server on every drag).
def monthly_payment_py(principal, annual_rate_pct, months):
    r = annual_rate_pct / 1200.0
    if r == 0.0:
        return principal / months
    factor = (1.0 + r) ** months
    return principal * r * factor / (factor - 1.0)
def pc_server(p):
    v = monthly_payment_py(float(p), 6.0, 360)
    return f'${v:,.2f} / month'

with gr.Blocks(title='pythscribe @wasm - client-side in a Gradio app') as app:
    gr.Markdown('## `@wasm` in your browser via `js=` - client-side, no round-trip')
    app.load(None, None, None, js=LOAD_JS)
    with gr.Tabs():
        with gr.Tab('Client-side filter'):
            gr.Markdown('Filter a loaded dataset of 2000 values **in the tab** - drag, no server hop.')
            f = gr.Slider(0, 1, value=0.5, step=0.01, label='threshold'); fo = gr.Textbox(label='result', interactive=False)
            f.change(None, [f], [fo], js=CJS('__filter'))
        with gr.Tab('Card validator (isomorphic)'):
            gr.Markdown('The same Luhn rule that would run on the server, run **in the tab** - the number never leaves.')
            v = gr.Textbox(value='4242 4242 4242 4242', label='card number'); vo = gr.Textbox(label='result', interactive=False)
            v.change(None, [v], [vo], js=CJS('__luhn'))
        with gr.Tab('On-device PII scan'):
            gr.Markdown('Scan for card/SSN/phone-like digit runs **in the tab** - the text is never uploaded.')
            pt = gr.Textbox(value='card 4242424242424242 ssn 123456789', label='text (stays in your browser)', lines=2); po = gr.Textbox(label='result', interactive=False)
            pt.change(None, [pt], [po], js=CJS('__pii'))
        with gr.Tab('Loan calculator'):
            gr.Markdown('Recomputes **with no server hop** as you drag - works offline.')
            cp = gr.Slider(10000, 1000000, value=300000, step=10000, label='principal ($)')
            cr = gr.Slider(0.5, 15, value=6, step=0.1, label='annual rate (%)')
            cm = gr.Slider(12, 480, value=360, step=12, label='term (months)')
            co = gr.Textbox(label='monthly payment', interactive=False)
            for sl in (cp, cr, cm): sl.change(None, [cp, cr, cm], [co], js=CJS('__calc'))
        with gr.Tab('Image filters (server, ~50x)'):
            gr.Markdown('Same filter, same input, two engines run **server-side** side by side: the '
                        'compiled `@wasm` kernel vs interpreted plain Python (the ~50x baseline). Drag '
                        'each threshold slider and compare the timing each column prints. (An in-tab '
                        'browser image path needs the typed-array adapter, v0.2.6.)')
            with gr.Row():
                with gr.Column():
                    ii = gr.Image(type='numpy', value=RGB0, height=240, label='input')
                    ik = gr.Radio(['threshold', 'sobel'], value='sobel', label='filter')
                with gr.Column():
                    gr.Markdown('### `@wasm` (compiled kernel)')
                    wsl = gr.Slider(0, 255, value=128, step=1, label='threshold')
                    iow = gr.Image(height=240, label='@wasm output')
                    itw = gr.Textbox(label='@wasm timing', interactive=False)
                with gr.Column():
                    gr.Markdown('### plain Python (interpreted)')
                    nsl = gr.Slider(0, 255, value=128, step=1, label='threshold')
                    ion = gr.Image(height=240, label='plain-Python output')
                    itn = gr.Textbox(label='plain-Python timing', interactive=False)
            wsl.release(lambda im, k, t: img_run(im, k, t, '@wasm'), [ii, ik, wsl], [iow, itw])
            nsl.release(lambda im, k, t: img_run(im, k, t, 'plain Python'), [ii, ik, nsl], [ion, itn])
            ik.change(lambda im, k, t: img_run(im, k, t, '@wasm'), [ii, ik, wsl], [iow, itw])
            ik.change(lambda im, k, t: img_run(im, k, t, 'plain Python'), [ii, ik, nsl], [ion, itn])
        with gr.Tab('Placement ceiling: browser vs server'):
            gr.Markdown('**Same compute (monthly payment @ 6% / 360 mo), two placements.** The *left* '
                        'slider recomputes **in your browser** via the compiled `@wasm` kernel (`js=`, 0 '
                        'server calls); the *right* slider is a native `gr.Slider` wired to a normal '
                        'Python `fn=` callback that **round-trips to the server on every drag**. Both are '
                        'honest choices - the native path is fine for lots of apps; the point is that '
                        'with `@wasm` the client-side recompute is *available* when you want it (latency, '
                        'offline, or keeping the input in the tab).')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### client-side `@wasm` (`js=`) - no round-trip')
                    pcp = gr.Slider(10000, 1000000, value=300000, step=10000, label='principal ($)')
                    pco = gr.Textbox(label='monthly payment', interactive=False)
                    pcp.change(None, [pcp], [pco], js=CJS('__pc'))
                with gr.Column():
                    gr.Markdown('### native server callback (`fn=`) - round-trips')
                    scp = gr.Slider(10000, 1000000, value=300000, step=10000, label='principal ($)')
                    sco = gr.Textbox(label='monthly payment', interactive=False)
                    (scp.change(pc_server, [scp], [sco],
                                js='(p) => { window.__st0 = performance.now(); return p; }')
                        .then(None, [sco], [sco],
                              js='(s) => s + "  [round-trip " + (performance.now() - window.__st0).toFixed(1) + " ms, to the server and back]"'))

app.launch(inbrowser=True)

## Metrics — the honest 4-way latency comparison (measured)

Two headless probes (subprocesses, so they work in Jupyter on Windows) drive a filter slider and time
the full slider→result latency for each path.

In [3]:
import json, subprocess, sys
root = (Path('..') / '..').resolve()
lp = subprocess.run([sys.executable, str(root / 'examples/wasm-use-cases/_latency_probe.py'), '2000'], capture_output=True, text=True, timeout=260)
bp = subprocess.run([sys.executable, str(root / 'examples/wasm-use-cases/_browser_only_probe.py'), '2000'], capture_output=True, text=True, timeout=120)
def _res(proc):
    l = next((x for x in proc.stdout.splitlines() if x.startswith('RESULT_JSON:')), None)
    return json.loads(l[len('RESULT_JSON:'):]) if l else None
r, b = _res(lp), _res(bp)
if not r:
    print('latency probe failed:\n', (lp.stderr or lp.stdout)[-1200:])
else:
    import pandas as pd
    base = r['clientjs_ms']
    rows = [
        {'path': 'static page (no Gradio)', 'latency_ms': round(b['median_ms'], 2) if b else None, 'round-trips': 0, 'input leaves browser': 'no'},
        {'path': 'client-side js= (in Gradio)', 'latency_ms': round(r['clientjs_ms'], 1), 'round-trips': 0, 'input leaves browser': 'no'},
        {'path': 'plain server', 'latency_ms': round(r['server_ms'], 1), 'round-trips': 1, 'input leaves browser': 'yes'},
        {'path': 'WasmFunction (old)', 'latency_ms': round(r['browser_ms'], 1), 'round-trips': 2, 'input leaves browser': 'yes'},
    ]
    df = pd.DataFrame(rows).set_index('path')
    print('interaction latency (slider -> result), median over a headless drive:')
    print(f"  the client-side js= path is ~{r['server_ms']/base:.0f}x faster than the server round-trip "
          f"and ~{r['browser_ms']/base:.0f}x faster than the old WasmFunction path -- with 0 round-trips and the input never leaving the browser.")
    display(df)

interaction latency (slider -> result), median over a headless drive:
  the client-side js= path is ~8x faster than the server round-trip and ~37x faster than the old WasmFunction path -- with 0 round-trips and the input never leaving the browser.


,latency_ms,round-trips,input leaves browser
path,,,
static page (no Gradio),0.1,0,no
client-side js= (in Gradio),7.9,0,no
plain server,63.8,1,yes
WasmFunction (old),294.8,2,yes


## Payload — the browser cost of keeping Python semantics

In [4]:
adir = binding_of(count_above).artifact.entry.parent
wasm_kb = (adir / 'count_above.wasm').stat().st_size / 1024
total_kb = sum(p.stat().st_size for p in adir.rglob('*') if p.is_file()) / 1024
print(f'@wasm bundle: {total_kb:.0f} KB  ({wasm_kb:.1f} KB .wasm + shared JS runtime; each extra kernel adds only its ~KB .wasm)')
print('Pyodide + NumPy (to keep Python semantics in the browser): ~14-19 MB, downloaded + compiled at page load.')

@wasm bundle: 427 KB  (0.5 KB .wasm + shared JS runtime; each extra kernel adds only its ~KB .wasm)
Pyodide + NumPy (to keep Python semantics in the browser): ~14-19 MB, downloaded + compiled at page load.


## Recap

- **`@wasm` runs client-side in a Gradio app today** via the built-in `js=` hook — **no custom
  component, no round-trip** — at **~10 ms** (vs ~60 ms server, ~290 ms WasmFunction), with the input
  never leaving the browser (so the PII/validator cases are genuinely private).
- **The `@wasm` compute itself is ~0.1 ms** (a plain static page hits that); the ~10 ms is Gradio's own
  client-side event overhead, not the compiler.
- **Remaining v0.2.6 items:** (A) a thin `pythscribe.gradio.client_side(kernel)` helper that generates
  this `load_js` + `change_js` wiring (so users don't hand-write it); (B) the **typed-array browser
  adapter** for *image* outputs (so the filters run in-tab too, not just server-side).

Server-side numeric comparison (Numba/NumPy) is in `full_features_demo.ipynb`; image preprocessing
(the bandwidth win that works today) is in `gradio_demo.ipynb`.